# MD Produccion 100ns - Celulasa Tr_Cel7A (Baroresistencia GH7)

**Sistema:** wt  
**Presion:** control (1.0 MPa)  
**Replica:** 1  
**GPU:** P100 (Kaggle free tier)  

**Pipeline:** Instalar deps → Subir PDB → Simular 100ns con checkpoints → Descargar resultados

## Celda 1: Instalar dependencias
OpenMM no viene preinstalado en Kaggle. Instalar con pip (~2 min).

In [ ]:
import subprocess, sys
# Instalar OpenMM via conda (mas confiable en Kaggle)
result = subprocess.run(
    ["pip", "install", "-q", "openmm", "mdtraj"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("pip fallo, intentando conda...")
    result = subprocess.run(
        ["conda", "install", "-y", "-c", "conda-forge", "openmm", "mdtraj"],
        capture_output=True, text=True
    )
if result.returncode != 0:
    print("ERROR instalando:")
    print(result.stderr[-500:])
    sys.exit(1)

import openmm
from openmm import Platform
print(f"OpenMM version: {openmm.__version__}")
platforms = [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())]
print(f"Plataformas: {platforms}")

## Celda 2: Subir archivo PDB solvatado
Sube wt_solvated.pdb (~4MB) usando el boton 'Upload' del panel Input o arrastra el archivo aqui.

In [ ]:
import os, base64
from openmm.app import PDBFile, Modeller, ForceField
import openmm.unit as unit

PDB_FILE = "wt_solvated.pdb"

# PDB embebido (clean_8CEL.pdb, 487KB)
pdb_b64 = "..."

# Escribir PDB desde base64
pdb_src = "/kaggle/working/clean_8CEL.pdb"
with open(pdb_src, "wb") as f:
    f.write(base64.b64decode(pdb_b64))
print("PDB escrito: " + pdb_src + " (" + str(round(os.path.getsize(pdb_src)/1024, 0)) + " KB)")

# Solvatacion inline
print("Solvatando con amber99sbildn + TIP3P (10A padding, 150 mM NaCl)...")
raw = PDBFile(pdb_src)
modeller = Modeller(raw.topology, raw.positions)
ff = ForceField("amber99sbildn.xml", "tip3p.xml")
modeller.addSolvent(ff, model="tip3p", padding=1.0*unit.nanometer, ionicStrength=0.15*unit.molar)
dst = "/kaggle/working/" + PDB_FILE
with open(dst, "w") as fout:
    PDBFile.writeFile(modeller.topology, modeller.positions, fout)
n_atoms = modeller.topology.getNumAtoms()
print("Sistema solvatado: " + str(n_atoms) + " atomos -> " + PDB_FILE)

## Celda 3: Definir script de simulacion inline
Todo el codigo de simulacion va aqui para evitar problemas de importacion de archivos externos.

In [ ]:
import sys, os, time, shutil, logging, glob
from pathlib import Path
from openmm.app import PDBFile, ForceField, Simulation, DCDReporter, StateDataReporter, PME, HBonds
from openmm import Platform, LangevinMiddleIntegrator, MonteCarloBarostat
import openmm.unit as unit

# ── Configuracion ──
SYSTEM = "wt"
PRESSURE = "control"
PRESSURE_VAL = 1.0  # bar
REPLICA = 1
NS_TO_RUN = 100.0
CHECKPOINT_INTERVAL_NS = 1.0

PREFIX = f"{SYSTEM}_{PRESSURE}_r{REPLICA}"
WORKDIR = Path("/kaggle/working")
PDB_PATH = WORKDIR / "wt_solvated.pdb"

# Logging
log_path = WORKDIR / f"{PREFIX}.log"
logger = logging.getLogger("md")
logger.setLevel(logging.INFO)
if not logger.handlers:
    fh = logging.FileHandler(str(log_path), mode="a")
    fh.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
    logger.addHandler(fh)
    ch = logging.StreamHandler(sys.stdout)
    ch.setFormatter(logging.Formatter("[%(levelname)s] %(message)s"))
    logger.addHandler(ch)

def safe_save(sim, chk_p, state_p):
    for p, bak_ext in [(chk_p, ".chk.bak"), (state_p, ".state.bak")]:
        if p.exists():
            shutil.copy2(str(p), str(p) + bak_ext)
    sim.saveCheckpoint(str(chk_p))
    sim.saveState(str(state_p))
    for p in [chk_p, state_p]:
        if not (p.exists() and p.stat().st_size > 0):
            raise RuntimeError(f"Checkpoint corrupto: {p}")
    logger.info(f"Checkpoint OK: {chk_p.name} ({chk_p.stat().st_size/1024:.0f} KB)")

def get_next_part(out_dir, prefix):
    parts = list(out_dir.glob(f"{prefix}_part*.dcd"))
    if not parts:
        return 1
    nums = []
    for f in parts:
        try:
            nums.append(int(f.name.split("_part")[1].split(".")[0]))
        except: pass
    return max(nums) + 1 if nums else 1

def run():
    logger.info(f"=== {SYSTEM.upper()} {PRESSURE.upper()} {PRESSURE_VAL} bar ===")
    
    # Plataforma
    platforms = [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())]
    logger.info(f"Plataformas: {platforms}")
    for pref in ["CUDA", "OpenCL", "CPU", "Reference"]:
        if pref in platforms:
            plat_name = pref
            props = {"Precision": "mixed"} if pref in ("CUDA", "OpenCL") else {}
            break
    platform = Platform.getPlatformByName(plat_name)
    logger.info(f"Usando: {plat_name}")
    
    # Cargar PDB
    logger.info(f"Cargando: {PDB_PATH}")
    pdb = PDBFile(str(PDB_PATH))
    
    # Sistema
    ff = ForceField("amber99sbildn.xml", "tip3p.xml")
    system = ff.createSystem(pdb.topology, nonbondedMethod=PME,
                             nonbondedCutoff=1.0*unit.nanometer, constraints=HBonds)
    barostat = MonteCarloBarostat(PRESSURE_VAL*unit.bar, 300.0*unit.kelvin, 25)
    barostat.setRandomNumberSeed(1000 + REPLICA)
    system.addForce(barostat)
    
    integrator = LangevinMiddleIntegrator(300.0*unit.kelvin, 1.0/unit.picosecond, 0.002*unit.picoseconds)
    integrator.setRandomNumberSeed(1000 + REPLICA)
    
    sim = Simulation(pdb.topology, system, integrator, platform, props)
    
    # Checkpoint paths
    chk_f = WORKDIR / f"{PREFIX}.chk"
    state_f = WORKDIR / f"{PREFIX}.state"
    
    # Reanudar?
    resumed = False
    if chk_f.exists():
        try:
            sim.loadCheckpoint(str(chk_f))
            logger.info(f"Reanudando desde paso {sim.currentStep}")
            resumed = True
        except Exception as e:
            logger.warning(f"Error .chk: {e}")
            if state_f.exists():
                try:
                    sim.loadState(str(state_f))
                    logger.info(f"Reanudado desde .state, paso {sim.currentStep}")
                    resumed = True
                except: pass
    
    if not resumed:
        logger.info("Nueva simulacion")
        sim.context.setPositions(pdb.positions)
        logger.info("Minimizando...")
        sim.minimizeEnergy(maxIterations=1000)
        sim.context.setVelocitiesToTemperature(300.0*unit.kelvin)
        logger.info("Equilibrando 1ns...")
        sim.step(500000)  # 1ns
        safe_save(sim, chk_f, state_f)
    
    # Produccion
    target_step = sim.currentStep + int(NS_TO_RUN * 1000 / 0.002)
    steps_remaining = target_step - sim.currentStep
    if steps_remaining <= 0:
        logger.info("Simulacion ya completa")
        return
    
    part_num = get_next_part(WORKDIR, PREFIX)
    dcd_f = WORKDIR / f"{PREFIX}_part{part_num:03d}.dcd"
    csv_f = WORKDIR / f"{PREFIX}_part{part_num:03d}.csv"
    report_interval = 50000  # 100ps
    chk_steps = int(CHECKPOINT_INTERVAL_NS * 1000 / 0.002)  # 1ns
    
    sim.reporters.append(DCDReporter(str(dcd_f), report_interval, append=False))
    sim.reporters.append(StateDataReporter(str(csv_f), report_interval,
        step=True, potentialEnergy=True, kineticEnergy=True, totalEnergy=True,
        temperature=True, density=True, volume=True, speed=True, append=False))
    
    logger.info(f"Produccion: {steps_remaining} pasos ({NS_TO_RUN} ns)")
    t0 = time.time()
    block = 0
    
    while steps_remaining > 0:
        block += 1
        n = min(chk_steps, steps_remaining)
        t1 = time.time()
        sim.step(n)
        dt = time.time() - t1
        steps_remaining -= n
        ns_day = (n * 0.002 / (dt/86400)) if dt > 0 else 0
        eta = (steps_remaining * 0.002/1000 / ns_day) if ns_day > 0 else 0
        logger.info(f"Bloque {block}: {ns_day:.1f} ns/dia | ETA {eta:.1f}d | Paso {sim.currentStep}")
        safe_save(sim, chk_f, state_f)
    
    elapsed = (time.time() - t0) / 3600
    logger.info(f"COMPLETADO: {elapsed:.1f}h")

run()

## Celda 4: Verificar resultados y descargar
Ejecutar despues de que termine la simulacion (o para verificar progreso).

In [ ]:
import os, glob
from pathlib import Path

prefix = PREFIX
print(f"Verificando: {prefix}")
print("=" * 50)

for ext in [".chk", ".chk.bak", ".state", ".state.bak"]:
    f = WORKDIR / f"{prefix}{ext}"
    if f.exists():
        print(f"  OK: {f.name} ({f.stat().st_size/1024:.0f} KB)")

dcds = sorted(glob.glob(str(WORKDIR / f"{prefix}_part*.dcd")))
total = 0
for d in dcds:
    sz = os.path.getsize(d) / 1024 / 1024
    total += sz
    print(f"  DCD: {os.path.basename(d)} ({sz:.1f} MB)")
print(f"  Total DCD: {total:.1f} MB en {len(dcds)} segmentos")

# Mostrar ultimas lineas del log
log_f = WORKDIR / f"{prefix}.log"
if log_f.exists():
    lines = log_f.read_text().splitlines()
    print(f"\n--- Ultimas 15 lineas del log ---")
    for l in lines[-15:]:
        print(l)

print("\nPara descargar: haz clic derecho en los archivos del panel Output -> Download")

## Celda 5: Descargar resultados (opcional)
Comprime todos los archivos de salida en un ZIP para descarga facil.

In [ ]:
import zipfile, glob, os

zip_path = f"/kaggle/working/{PREFIX}_results.zip"
patterns = [
    f"{PREFIX}*.dcd",
    f"{PREFIX}*.csv",
    f"{PREFIX}.chk",
    f"{PREFIX}.state",
    f"{PREFIX}.log",
]

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for pat in patterns:
        for f in glob.glob(f"/kaggle/working/{pat}"):
            zf.write(f, os.path.basename(f))
            print(f"  Anadido: {os.path.basename(f)}")

size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f"\nZIP creado: {zip_path} ({size_mb:.1f} MB)")
print("Descarga desde el panel Output (archivos de trabajo)")